# Creación de metadatos de videos de plank

Esta notebook recorre `data/bronze/videos/plank/`, obtiene los metadatos de cada video y crea `data/bronze/metadata/plank_metadata.csv`.

El nombre esperado sigue el patrón `good_plank_XX_pX_camera-view.mp4` o `bad_plank_XX_pX_camera-view.mp4`. Por ejemplo, `good_plank_03_p3_back-side.mp4`. El número `XX` identifica la grabación dentro del nombre original; `video_id` se asigna de forma secuencial para que cada fila tenga un identificador único.

La carpeta padre (`correct` o `incorrect`) es la fuente de verdad para `label`. El prefijo `good`/`bad` del nombre también se valida para detectar archivos mal ubicados. Si el nombre no incluye `pX`, `Participant_id` queda vacío: no se infiere un participante que no esté indicado.

## 1. Importaciones y rutas

Se usa OpenCV para leer los FPS y el número de cuadros del contenedor de video. La duración se calcula como `número de cuadros / FPS`. Si no se puede abrir un archivo o sus propiedades no son válidas, la notebook se detiene con un error explícito para evitar escribir metadatos incompletos.

In [1]:
from pathlib import Path
import math
import re

import cv2
import pandas as pd

# Permite ejecutar la notebook desde la raíz del proyecto o desde notebooks/.
PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "data" / "bronze" / "videos" / "plank").is_dir():
    PROJECT_ROOT = PROJECT_ROOT.parent

VIDEO_ROOT = PROJECT_ROOT / "data" / "bronze" / "videos" / "plank"
OUTPUT_PATH = PROJECT_ROOT / "data" / "bronze" / "metadata" / "plank_metadata.csv"

if not VIDEO_ROOT.is_dir():
    raise FileNotFoundError(f"No se encontró la carpeta de videos: {VIDEO_ROOT}")

print(f"Carpeta de videos: {VIDEO_ROOT}")
print(f"Archivo de salida: {OUTPUT_PATH}")

Carpeta de videos: d:\Proyectos_personales\KineSight\data\bronze\videos\plank
Archivo de salida: d:\Proyectos_personales\KineSight\data\bronze\metadata\plank_metadata.csv


## 2. Descubrir y validar los videos

Los archivos se procesan primero por etiqueta y luego por nombre, ambos en orden alfabético. Así, al repetir la ejecución con el mismo conjunto de videos, los `video_id` se asignan en el mismo orden. La expresión regular permite que el identificador de participante sea opcional y conserva vistas como `back-side`.

In [2]:
VIDEO_NAME_PATTERN = re.compile(
    r"^(?P<quality>good|bad)_(?P<exercise>[^_]+)_"
    r"(?P<recording_number>\d+)_"
    r"(?:(?P<participant_id>p\d+)_)?"
    r"(?P<camera_view>.+)$",
    re.IGNORECASE,
)
VIDEO_EXTENSIONS = {".mp4", ".mov", ".avi", ".mkv", ".mpeg", ".mpg"}
FOLDER_LABELS = {"correct": "correct", "incorrect": "incorrect"}
QUALITY_LABELS = {"good": "correct", "bad": "incorrect"}

video_files = []
for folder_name in FOLDER_LABELS:
    label_folder = VIDEO_ROOT / folder_name
    if not label_folder.is_dir():
        raise FileNotFoundError(f"No se encontró la carpeta de etiquetas: {label_folder}")
    video_files.extend(
        path
        for path in label_folder.rglob("*")
        if path.is_file() and path.suffix.lower() in VIDEO_EXTENSIONS
    )

video_files.sort(key=lambda path: (path.parent.name.lower(), path.name.lower()))
if not video_files:
    raise FileNotFoundError(f"No se encontraron videos compatibles en {VIDEO_ROOT}")

parsed_videos = []
for video_path in video_files:
    match = VIDEO_NAME_PATTERN.fullmatch(video_path.stem)
    if match is None:
        raise ValueError(f"El nombre no sigue el formato esperado: {video_path.name}")

    label = FOLDER_LABELS.get(video_path.parent.name.lower())
    if label is None:
        raise ValueError(f"El video no está dentro de correct/incorrect: {video_path}")

    filename_label = QUALITY_LABELS[match.group("quality").lower()]
    if filename_label != label:
        raise ValueError(
            f"La etiqueta del nombre no coincide con su carpeta: {video_path}"
        )

    parsed_videos.append(
        {
            "path": video_path,
            "Participant_id": (
                match.group("participant_id").upper()
                if match.group("participant_id")
                else ""
            ),
            "excercise": match.group("exercise").lower(),
            "label": label,
            "camera_view": match.group("camera_view"),
        }
    )

print(f"Videos encontrados y validados: {len(parsed_videos)}")

Videos encontrados y validados: 14


## 3. Leer propiedades y construir la tabla

`video_id` usa el formato `P01`, `P02`, etc. Es un identificador de video y no debe confundirse con `Participant_id`, que se extrae del nombre (por ejemplo, `p3` se normaliza a `P3`). Duración y FPS se guardan con tres decimales para evitar precisión aparente que no aportan los metadatos del video.

In [3]:
rows = []
for index, video in enumerate(parsed_videos, start=1):
    capture = cv2.VideoCapture(str(video["path"]))
    try:
        if not capture.isOpened():
            raise OSError(f"OpenCV no pudo abrir el video: {video['path']}")

        frame_count = capture.get(cv2.CAP_PROP_FRAME_COUNT)
        fps = capture.get(cv2.CAP_PROP_FPS)
        if (
            not math.isfinite(frame_count)
            or not math.isfinite(fps)
            or frame_count <= 0
            or fps <= 0
        ):
            raise ValueError(
                f"Duración/FPS invalidos para {video['path'].name}: "
                f"cuadros={frame_count}, FPS={fps}"
            )

        rows.append(
            {
                "video_id": f"P{index:02d}",
                "Participant_id": video["Participant_id"],
                "excercise": video["excercise"],
                "label": video["label"],
                "camera_view": video["camera_view"],
                "duration_in_seconds": round(frame_count / fps, 3),
                "fps": round(fps, 3),
            }
        )
    finally:
        capture.release()

columns = [
    "video_id",
    "Participant_id",
    "excercise",
    "label",
    "camera_view",
    "duration_in_seconds",
    "fps",
]
metadata = pd.DataFrame(rows, columns=columns)
metadata

,video_id,Participant_id,excercise,label,camera_view,duration_in_seconds,fps
0,P01,P1,plank,correct,side,2.633,30.0
1,P02,P2,plank,correct,side,5.567,30.0
2,P03,P3,plank,correct,back-side,1.733,30.0
3,P04,P5,plank,correct,side,3.000,30.0
4,P05,P6,plank,correct,side,2.733,30.0
5,P06,P6,plank,correct,side,3.767,30.0
6,P07,P7,plank,correct,side,3.033,30.0
7,P08,P1,plank,incorrect,side,4.567,30.0
8,P09,P2,plank,incorrect,side,1.867,30.0
9,P10,P4,plank,incorrect,side,1.067,30.0


## 4. Guardar el CSV

El CSV se escribe en la carpeta de metadata del nivel bronze, usando UTF-8 y sin una columna índice adicional. Las filas con participante desconocido se representan con un campo vacío.

In [4]:
OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
metadata.to_csv(OUTPUT_PATH, index=False, encoding="utf-8")

print(f"CSV creado: {OUTPUT_PATH}")
print(f"Filas: {len(metadata)} | Columnas: {len(metadata.columns)}")
print(f"Participantes sin indicar: {(metadata['Participant_id'] == '').sum()}")

CSV creado: d:\Proyectos_personales\KineSight\data\bronze\metadata\plank_metadata.csv
Filas: 14 | Columnas: 7
Participantes sin indicar: 0
